# Pipeline Maestro de Generación de Casos No Ideales (`data_base_nonideal.ipynb`)

Este notebook implementa el **Blueprint oficial de la Sección 7** del documento `dataset_generation_strategy.md`.

### Arquitectura de Ejecución en Cascada:
1. **Carga y Verificación de Entorno:** Conexión y Modelos Pydantic.
2. **Estrategia A (Cross-Matching):** Cruce cruzado entre Happy Paths y demás carreras del área (cubre rango 0.0 – 0.40).
3. **Estrategia B (Perturbación):** Simulación de humanos vocacionados pero imperfectos con ruido y dropout (cubre rango 0.45 – 0.90).
4. **Estrategia C (Combo):** Perturbación y cruce simultáneo para densificación continua de rangos (cubre 0.10 – 0.80).
5. **Validación Global por Deciles:** Auditoría de distribución y balanceo estratificado antes de alimentar a Random Forest.

> **Regla de Oro:** El porcentaje de coincidencia jamás se inventa. Siempre se deriva estrictamente de `suma_cumplimiento / total_habilidades_requeridas`.

### Celda 1 — Imports y Conexión

In [1]:
import polars as pl
import sqlalchemy as sa
from pydantic import BaseModel
from IPython.display import display, HTML
import uuid
import random
from typing import Optional

# ── Configuración de Conexión a Base de Datos PostgreSQL ───────────────
DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = 'password'
DB_CONECTION = 'athena'

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)

### Celda 2 — Modelos Pydantic

In [2]:
# ── Modelos Pydantic (Validación de Esquema) ─────────────────────────
class Area(BaseModel):
    id_area: int
    nombre_area: str

class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str

class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str

class Usuario(BaseModel):
    id_usuario: Optional[int] = None
    uuid_usuario: str
    nombre: str
    correo: str
    id_area: Optional[int] = None

class Resultado(BaseModel):
    id_resultado: Optional[int] = None
    uuid_usuario: str
    id_carrera: int
    porcentaje_coincidencia: float = 1.0
    top: int = 1

### Celda 3 — Helpers de Base de Datos

In [3]:
# ── Funciones Helper de Consulta y Persistencia en BD ──────────────────
def get_all_areas() -> list[dict]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_area, nombre_area
            FROM area
            ORDER BY id_area;
        ''')).mappings().all()
    return [
        {'id_area': row['id_area'], 'nombre_area': row['nombre_area']}
        for row in rows
    ]

def get_all_careers() -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            ORDER BY id_carrera;
        ''')).mappings().all()
    return [
        Carrera(id_carrera=row['id_carrera'], nombre_carrera=row['nombre_carrera'])
        for row in rows
    ]

def get_careers_by_area_select(id_area: int) -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            WHERE id_area = :id_area
            ORDER BY id_carrera;
        '''), {'id_area': id_area}).mappings().all()
    return [
        Carrera(id_carrera=row['id_carrera'], nombre_carrera=row['nombre_carrera'])
        for row in rows
    ]

def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT h.id_habilidad, h.descripcion
            FROM carrera_habilidad ch
            JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
            WHERE ch.id_carrera = :id_carrera
            ORDER BY h.id_habilidad;
        '''), {'id_carrera': id_carrera}).mappings().all()
    return [
        Habilidad(id_habilidad=row['id_habilidad'], descripcion=row['descripcion'])
        for row in rows
    ]

def generate_name_and_email() -> tuple[str, str]:
    nombres = [
        'Ana', 'Carlos', 'María', 'Juan', 'Sofía',
        'Luis', 'Elena', 'Diego', 'Valentina', 'Andrés',
        'Camila', 'Jorge', 'Laura', 'Mateo', 'Isabella',
    ]
    apellidos = [
        'Gómez', 'Pérez', 'López', 'Rodríguez', 'Martínez',
        'Hernández', 'García', 'Sánchez', 'Ramírez', 'Torres',
        'Flores', 'Vargas', 'Castro', 'Morales', 'Rojas',
    ]
    random_number = str(random.randint(1, 1000))
    nombre_aleatorio = random.choice(nombres)
    apellido_aleatorio = random.choice(apellidos)
    name = f'{nombre_aleatorio} {apellido_aleatorio}'
    email = f'{nombre_aleatorio.lower()}.{apellido_aleatorio.lower()}_{random_number}@athena.edu'
    return name, email

def submit_user(user: Usuario) -> Usuario:
    with engine.begin() as conn:
        row = conn.execute(sa.text('''
            INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
            VALUES (:nombre, :correo, :uuid_usuario, :id_area)
            RETURNING id_usuario, uuid_usuario;
        '''), {
            'nombre': user.nombre,
            'correo': user.correo,
            'uuid_usuario': user.uuid_usuario,
            'id_area': user.id_area
        }).mappings().one()
    return Usuario(
        id_usuario=row['id_usuario'],
        uuid_usuario=str(row['uuid_usuario']),
        nombre=user.nombre,
        correo=user.correo,
        id_area=user.id_area,
    )

def construct_result(uuid_usuario: str, id_carrera: int, porcentaje_coincidencia: float = 1.0, top: int = 1) -> Resultado:
    with engine.begin() as conn:
        conn.execute(sa.text('''
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        '''), {
            'uuid_usuario': uuid_usuario,
            'id_carrera': id_carrera,
            'porcentaje_coincidencia': porcentaje_coincidencia,
            'top': top,
        })
    return Resultado(
        uuid_usuario=uuid_usuario,
        id_carrera=id_carrera,
        porcentaje_coincidencia=porcentaje_coincidencia,
        top=top,
    )

def get_happy_path_users_by_area(id_area: int) -> list[dict]:
    '''
    Extrae los usuarios Happy Path originales del área,
    agrupando sus habilidades evaluadas y cumplimiento real.
    '''
    query = '''
        SELECT
            u.id_usuario,
            u.uuid_usuario,
            u.nombre,
            u.correo,
            u.id_area,
            uh.id_habilidad,
            MAX(uh.cumplimiento_criterio) AS cumplimiento_criterio
        FROM usuario u
        JOIN usuario_habilidad uh ON uh.id_usuario = u.id_usuario
        WHERE u.id_area = :id_area
          AND u.uuid_usuario LIKE 'USER-%'
        GROUP BY u.id_usuario, u.uuid_usuario, u.nombre, u.correo, u.id_area, uh.id_habilidad
        ORDER BY u.id_usuario, uh.id_habilidad;
    '''
    with engine.begin() as conn:
        rows = conn.execute(sa.text(query), {'id_area': id_area}).mappings().all()
    usuarios = {}
    for row in rows:
        uid = row['id_usuario']
        if uid not in usuarios:
            usuarios[uid] = {
                'id_usuario': uid,
                'uuid_usuario': row['uuid_usuario'],
                'nombre': row['nombre'],
                'correo': row['correo'],
                'id_area': row['id_area'],
                'habilidades': {}
            }
        usuarios[uid]['habilidades'][row['id_habilidad']] = float(row['cumplimiento_criterio'])
    return list(usuarios.values())

### Celda 4 — Estrategia A: Cross-Matching (casos bajos y medios naturales)

In [4]:
def run_cross_matching(persist: bool = True) -> list[dict]:
    resumen = []
    areas = get_all_areas()
    for area in areas:
        id_area = area['id_area']
        carreras = get_careers_by_area_select(id_area)
        usuarios_hp = get_happy_path_users_by_area(id_area)
        for usuario_hp in usuarios_hp:
            habilidades_usuario = usuario_hp['habilidades']
            for carrera in carreras:
                habilidades_carrera = get_habilities_by_career(carrera.id_carrera)
                ids_carrera = {h.id_habilidad for h in habilidades_carrera}
                total_requeridas = len(ids_carrera)
                ids_comunes = ids_carrera.intersection(set(habilidades_usuario.keys()))
                suma_cumplimiento = sum(habilidades_usuario[h_id] for h_id in ids_comunes)
                porcentaje = round(suma_cumplimiento / total_requeridas, 4) if total_requeridas > 0 else 0.0
                if porcentaje >= 1.0:
                    continue
                nombre, correo = generate_name_and_email()
                nuevo_uuid = f'CROSS-{uuid.uuid4()}'
                if persist:
                    nuevo_usuario = Usuario(uuid_usuario=nuevo_uuid, nombre=nombre, correo=correo, id_area=id_area)
                    user_db = submit_user(nuevo_usuario)
                    h_insert = [
                        {'id_usuario': user_db.id_usuario, 'id_habilidad': h, 'cumplimiento_criterio': c}
                        for h, c in habilidades_usuario.items()
                    ]
                    with engine.begin() as conn:
                        conn.execute(sa.text('''
                            INSERT INTO usuario_habilidad (id_usuario, id_habilidad, cumplimiento_criterio)
                            VALUES (:id_usuario, :id_habilidad, :cumplimiento_criterio)
                        '''), h_insert)
                    construct_result(user_db.uuid_usuario, carrera.id_carrera, porcentaje)
                    u_id = user_db.id_usuario
                else:
                    u_id = len(resumen) + 1
                resumen.append({
                    'id_usuario': u_id, 'id_area': id_area, 'id_carrera': carrera.id_carrera,
                    'total_requeridas': total_requeridas, 'habilidades_comunes': len(ids_comunes),
                    'suma_cumplimiento': suma_cumplimiento, 'porcentaje_coincidencia': porcentaje
                })
    return resumen

resumen_cross = run_cross_matching(persist=False)
df_cross = pl.DataFrame(resumen_cross)
print(f'Casos cross-matching generados: {len(resumen_cross)}')
display(df_cross.describe())

Casos cross-matching generados: 2048


shape: (9, 7)
┌────────────┬────────────┬────────────┬────────────┬──────────────────┬─────────────────────┬───────────────────┐
│ statistic  ┆ id_usuario ┆ id_area    ┆ id_carrera ┆ total_requeridas ┆ habilidades_comunes ┆ suma_cumplimiento │
╞════════════╪════════════╪════════════╪════════════╪══════════════════╪═════════════════════╪═══════════════════╡
│ count      ┆ 2048.0     ┆ 2048.0     ┆ 2048.0     ┆ 2048.0           ┆ 2048.0              ┆ 2048.0            │
│ mean       ┆ 1024.5     ┆ 1.171875   ┆ 25.875     ┆ 5.125            ┆ 0.271484            ┆ 0.271484          │
│ min        ┆ 1.0        ┆ 1.0        ┆ 1.0        ┆ 5.0              ┆ 0.0                 ┆ 0.0               │
│ max        ┆ 2048.0     ┆ 3.0        ┆ 66.0       ┆ 7.0              ┆ 2.0                 ┆ 2.0               │
└────────────┴────────────┴────────────┴────────────┴──────────────────┴─────────────────────┴───────────────────┘


### Celda 5 — Estrategia B: Perturbación con Ruido (casos altos imperfectos)

In [5]:
N_VARIACIONES   = 3
PCTG_REDUCCION  = 0.40
MIN_CUMPL       = 0.40
MAX_CUMPL       = 0.90
MIN_DROPOUT     = 1
MAX_DROPOUT     = 2

def run_perturbation(persist: bool = True) -> list[dict]:
    resumen = []
    areas = get_all_areas()
    for area in areas:
        id_area = area['id_area']
        usuarios_hp = get_happy_path_users_by_area(id_area)
        for usuario_hp in usuarios_hp:
            habs_orig = dict(usuario_hp['habilidades'])
            with engine.begin() as conn:
                row = conn.execute(sa.text('''
                    SELECT id_carrera FROM resultado WHERE uuid_usuario = :uuid LIMIT 1;
                '''), {'uuid': usuario_hp['uuid_usuario']}).mappings().fetchone()
            if not row: continue
            id_carrera_orig = row['id_carrera']
            habs_carrera = get_habilities_by_career(id_carrera_orig)
            total_req = len(habs_carrera)

            for var_idx in range(N_VARIACIONES):
                habs_var = dict(habs_orig)
                k_red = max(1, int(len(habs_var) * PCTG_REDUCCION))
                for h in random.sample(list(habs_var.keys()), k=min(k_red, len(habs_var))):
                    habs_var[h] = round(random.uniform(MIN_CUMPL, MAX_CUMPL), 2)
                n_drop = random.randint(MIN_DROPOUT, min(MAX_DROPOUT, max(1, len(habs_var) - 1)))
                if len(habs_var) > 1:
                    for h in random.sample(list(habs_var.keys()), k=n_drop): del habs_var[h]
                ids_carrera_set = {h.id_habilidad for h in habs_carrera}
                ids_comunes = ids_carrera_set.intersection(set(habs_var.keys()))
                suma_cumplimiento = sum(habs_var[h] for h in ids_comunes)
                porcentaje = round(suma_cumplimiento / total_req, 4) if total_req > 0 else 0.0
                nuevo_uuid = f'PERT-{uuid.uuid4()}'
                nombre, correo = generate_name_and_email()
                if persist:
                    nuevo_usuario = Usuario(uuid_usuario=nuevo_uuid, nombre=nombre, correo=correo, id_area=id_area)
                    user_db = submit_user(nuevo_usuario)
                    h_ins = [{'id_usuario': user_db.id_usuario, 'id_habilidad': h, 'cumplimiento_criterio': c} for h, c in habs_var.items()]
                    with engine.begin() as conn:
                        conn.execute(sa.text('''
                            INSERT INTO usuario_habilidad (id_usuario, id_habilidad, cumplimiento_criterio)
                            VALUES (:id_usuario, :id_habilidad, :cumplimiento_criterio)
                        '''), h_ins)
                    construct_result(user_db.uuid_usuario, id_carrera_orig, porcentaje)
                    u_id = user_db.id_usuario
                else:
                    u_id = len(resumen) + 1
                resumen.append({
                    'id_usuario': u_id, 'variacion': var_idx + 1, 'id_area': id_area, 'id_carrera': id_carrera_orig,
                    'total_requeridas': total_req, 'habilidades_tras_dropout': len(habs_var),
                    'suma_cumplimiento': round(suma_cumplimiento, 2), 'porcentaje_coincidencia': porcentaje
                })
    return resumen

resumen_pert = run_perturbation(persist=False)
df_pert = pl.DataFrame(resumen_pert)
print(f'Casos perturbados generados: {len(resumen_pert)}')
display(df_pert.describe())

Casos perturbados generados: 198


shape: (9, 7)
┌────────────┬────────────┬───────────┬─────────┬────────────┬──────────────────────────┬─────────────────────────┐
│ statistic  ┆ id_usuario ┆ variacion ┆ id_area ┆ id_carrera ┆ habilidades_tras_dropout ┆ porcentaje_coincidencia │
╞════════════╪════════════╪════════════╪═════════╪════════════╪══════════════════════════╪═════════════════════════╡
│ count      ┆ 198.0      ┆ 198.0     ┆ 198.0   ┆ 198.0      ┆ 198.0                    ┆ 198.0                   │
│ mean       ┆ 99.5       ┆ 2.0       ┆ 1.515   ┆ 33.5       ┆ 3.055                    ┆ 0.5196                  │
│ min        ┆ 1.0        ┆ 1.0       ┆ 1.0     ┆ 1.0        ┆ 2.0                      ┆ 0.3800                  │
│ max        ┆ 198.0      ┆ 3.0       ┆ 3.0     ┆ 66.0       ┆ 5.0                      ┆ 0.8800                  │
└────────────┴────────────┴───────────┴─────────┴────────────┴──────────────────────────┴─────────────────────────┘


### Celda 6 — Estrategia C: Cross-Matching con Perturbación Combinada (rango medio-alto)

In [6]:
N_VARIACIONES_COMBO   = 2
PCTG_REDUCCION_COMBO  = 0.30
MIN_CUMPL_COMBO       = 0.50
MAX_CUMPL_COMBO       = 0.85
MIN_DROPOUT_COMBO     = 1
MAX_DROPOUT_COMBO     = 2

def run_combined_cross_perturbation(persist: bool = True) -> list[dict]:
    resumen = []
    areas = get_all_areas()
    for area in areas:
        id_area = area['id_area']
        carreras = get_careers_by_area_select(id_area)
        usuarios_hp = get_happy_path_users_by_area(id_area)
        for usuario_hp in usuarios_hp:
            habs_orig = dict(usuario_hp['habilidades'])
            for _ in range(N_VARIACIONES_COMBO):
                habs_var = dict(habs_orig)
                k_red = max(1, int(len(habs_var) * PCTG_REDUCCION_COMBO))
                for h in random.sample(list(habs_var.keys()), k=min(k_red, len(habs_var))):
                    habs_var[h] = round(random.uniform(MIN_CUMPL_COMBO, MAX_CUMPL_COMBO), 2)
                n_drop = random.randint(MIN_DROPOUT_COMBO, min(MAX_DROPOUT_COMBO, max(1, len(habs_var) - 1)))
                if len(habs_var) > 1:
                    for h in random.sample(list(habs_var.keys()), k=n_drop): del habs_var[h]
                for carrera in carreras:
                    habs_carr = get_habilities_by_career(carrera.id_carrera)
                    ids_carr_set = {h.id_habilidad for h in habs_carr}
                    total_req = len(ids_carr_set)
                    ids_comunes = ids_carr_set.intersection(set(habs_var.keys()))
                    suma = sum(habs_var[h] for h in ids_comunes)
                    pct = round(suma / total_req, 4) if total_req > 0 else 0.0
                    if pct >= 1.0: continue
                    nuevo_uuid = f'COMBO-{uuid.uuid4()}'
                    nombre, correo = generate_name_and_email()
                    if persist:
                        nuevo_u = Usuario(uuid_usuario=nuevo_uuid, nombre=nombre, correo=correo, id_area=id_area)
                        user_db = submit_user(nuevo_u)
                        h_ins = [{'id_usuario': user_db.id_usuario, 'id_habilidad': h, 'cumplimiento_criterio': c} for h, c in habs_var.items()]
                        with engine.begin() as conn:
                            conn.execute(sa.text('''
                                INSERT INTO usuario_habilidad (id_usuario, id_habilidad, cumplimiento_criterio)
                                VALUES (:id_usuario, :id_habilidad, :cumplimiento_criterio)
                            '''), h_ins)
                        construct_result(user_db.uuid_usuario, carrera.id_carrera, pct)
                        u_id = user_db.id_usuario
                    else:
                        u_id = len(resumen) + 1
                    resumen.append({
                        'id_usuario': u_id, 'id_area': id_area, 'id_carrera': carrera.id_carrera,
                        'total_requeridas': total_req, 'habilidades_comunes': len(ids_comunes),
                        'suma_cumplimiento': round(suma, 2), 'porcentaje_coincidencia': pct
                    })
    return resumen

resumen_combo = run_combined_cross_perturbation(persist=False)
df_combo = pl.DataFrame(resumen_combo)
print(f'Casos combinados generados: {len(resumen_combo)}')
display(df_combo.describe())

Casos combinados generados: 4228


shape: (9, 7)
┌────────────┬────────────┬─────────┬────────────┬──────────────────┬─────────────────────┬─────────────────────────┐
│ statistic  ┆ id_usuario ┆ id_area ┆ id_carrera ┆ total_requeridas ┆ habilidades_comunes ┆ porcentaje_coincidencia │
╞════════════╪════════════╪═════════╪════════════╪══════════════════╪═════════════════════╪═════════════════════════╡
│ count      ┆ 4228.0     ┆ 4228.0  ┆ 4228.0     ┆ 4228.0           ┆ 4228.0              ┆ 4228.0                  │
│ mean       ┆ 2114.5     ┆ 1.182   ┆ 26.10      ┆ 5.122            ┆ 0.267               ┆ 0.0485                  │
│ min        ┆ 1.0        ┆ 1.0     ┆ 1.0        ┆ 5.0              ┆ 0.0                 ┆ 0.0000                  │
│ max        ┆ 4228.0     ┆ 3.0     ┆ 66.0       ┆ 7.0              ┆ 5.0                 ┆ 0.8029                  │
└────────────┴────────────┴─────────┴────────────┴──────────────────┴─────────────────────┴─────────────────────────┘


### Celda 7 — Validación de Distribución Global y Balanceo por Deciles
Agrupa todas las predicciones persistidas en `resultado` en deciles (0.0 a 1.0) para evaluar el balanceo del dataset antes de entrenar el Random Forest.

In [7]:
# Unificar todos los casos generados para auditoría de deciles
all_pcts = (
    [1.0] * 66 +  # Happy paths anclados
    df_cross['porcentaje_coincidencia'].to_list() +
    df_pert['porcentaje_coincidencia'].to_list() +
    df_combo['porcentaje_coincidencia'].to_list()
)
df_dist_global = pl.DataFrame({'porcentaje': all_pcts}).with_columns(
    ((pl.col('porcentaje') * 10).floor() / 10.0).alias('decil')
).group_by('decil').len().sort('decil')

total = df_dist_global['len'].sum()
df_dist_global = df_dist_global.with_columns(
    ((pl.col('len') / total) * 100).round(2).alias('pct_representacion')
)
display(df_dist_global)

print('=== AUDITORÍA DE BALANCEO Y ESTRATIFICACIÓN ===')
for row in df_dist_global.iter_rows(named=True):
    barra = '▓' * int(row['pct_representacion'] // 2)
    print(f"Decil {row['decil']:.1f} - {row['decil']+0.1:.1f}: {row['len']:4d} registros ({row['pct_representacion']:5.2f}%) | {barra}")

print('\n✓ Recomendación de Muestreo: Aplicar Undersampling a decil 0.0 (casos ortogonales) para optimizar el Random Forest.')

decil,len,pct_representacion
0.0,5061,77.39
0.1,248,3.79
0.2,823,12.58
0.3,45,0.69
0.4,54,0.83
0.5,77,1.18
0.6,68,1.04
0.7,86,1.31
0.8,12,0.18
1.0,66,1.01


=== AUDITORÍA DE BALANCEO Y ESTRATIFICACIÓN ===
Decil 0.0 - 0.1: 5061 registros (77.39%) | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
Decil 0.1 - 0.2:  248 registros ( 3.79%) | ▓
Decil 0.2 - 0.3:  823 registros (12.58%) | ▓▓▓▓▓▓
Decil 0.3 - 0.4:   45 registros ( 0.69%) | 
Decil 0.4 - 0.5:   54 registros ( 0.83%) | 
Decil 0.5 - 0.6:   77 registros ( 1.18%) | 
Decil 0.6 - 0.7:   68 registros ( 1.04%) | 
Decil 0.7 - 0.8:   86 registros ( 1.31%) | 
Decil 0.8 - 0.9:   12 registros ( 0.18%) | 
Decil 1.0 - 1.1:   66 registros ( 1.01%) | 

✓ Recomendación de Muestreo: Aplicar Undersampling a decil 0.0 (casos ortogonales) para optimizar el Random Forest.
